# Preprocesamiento de Datos

**Objetivo principal** : Preparar los datos de los subyacentes y las cadenas de opciones para poder calcular precios teóricos de opciones mediante Monte Carlo, Árbol Binomial y Black-Scholes. Esto incluye limpieza, filtrado, cálculo de parámetros históricos y selección de los contratos apropiados.

## Preparación del dataset de subyacentes

In [1]:
import pandas as pd
df_hist = pd.read_pickle("../data/historico_desde_2024_trabajo_MNFI.pkl")

In [2]:
df_hist.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 531 entries, 2024-01-02 to 2026-01-21
Columns: 162 entries, ('Adj Close', 'AAPL') to ('Volume', 'WBD')
dtypes: float64(162)
memory usage: 676.2 KB


In [3]:
df_hist.columns.levels[0]


Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

Imprimimos de antemano información general del dataset que nos han facilitado, observamos que tiene datos desde `2024-01-02` a `2026-01-21`.

Para cada activo se nos da información de `Adj Close` , `Close`,`High`,`Low`,`Open` y`Volume`:
- **Adjusted Close Price**: Es el precio de cierre de la acción, pero ajustado por dividendos, splits y otros eventos corporativos. Clave para estimar rentabilidades y volatilidad histórica.

- **Close**: Precio de cierre sin ajustes. Representa el último precio al que se negoció el activo durante la sesión. 

- **Open**: Precio al que el activo comenzó a cotizar al inicio de la sesión.  

- **High**: Precio máximo alcanzado durante la sesión.  
  Indica niveles de resistencia y la presión compradora durante el día.

- **Low**:Precio mínimo alcanzado durante la sesión.  

- **Volume** : Volumen de operaciones del día de dicho activo, con dicho dato podemos observar cambio de tendencia o liquidez




In [4]:

nan_totales = df_hist.isna().sum().sum()  
nan_por_col = df_hist.isna().sum()

nan_top10 = nan_por_col.sort_values(ascending=False).head(10)


duplicados = df_hist.duplicated().sum()
filas_duplicadas = df_hist[df_hist.duplicated(keep=False)] if duplicados > 0 else pd.DataFrame()


if isinstance(df_hist.index, pd.DatetimeIndex):
    fechas_esperadas = pd.date_range(start=df_hist.index.min(), end=df_hist.index.max(), freq='B')
    fechas_faltantes = fechas_esperadas.difference(df_hist.index)
else:
    fechas_faltantes = []

'''
print("="*50)
print("INTEGRIDAD DE DATOS - RESUMEN")
print("="*50)
print(f"Total de valores NaN en el DataFrame: {nan_totales}\n")

if not nan_top10.empty:
    print("Top 10 columnas con más valores NaN:")
    print(nan_top10)
else:
    print("No hay valores NaN en ninguna columna.\n")

print("-"*50)
print(f"Número de filas duplicadas: {duplicados}")
if duplicados > 0:
    print("Filas duplicadas:")
    print(filas_duplicadas)
print("-"*50)

if len(fechas_faltantes) > 0:
    print(f"Fechas faltantes ({len(fechas_faltantes)}):")
    print(fechas_faltantes)
else:
    print("No hay fechas faltantes.\n")
print("="*50)
'''


'\nprint("="*50)\nprint("INTEGRIDAD DE DATOS - RESUMEN")\nprint("="*50)\nprint(f"Total de valores NaN en el DataFrame: {nan_totales}\n")\n\nif not nan_top10.empty:\n    print("Top 10 columnas con más valores NaN:")\n    print(nan_top10)\nelse:\n    print("No hay valores NaN en ninguna columna.\n")\n\nprint("-"*50)\nprint(f"Número de filas duplicadas: {duplicados}")\nif duplicados > 0:\n    print("Filas duplicadas:")\n    print(filas_duplicadas)\nprint("-"*50)\n\nif len(fechas_faltantes) > 0:\n    print(f"Fechas faltantes ({len(fechas_faltantes)}):")\n    print(fechas_faltantes)\nelse:\n    print("No hay fechas faltantes.\n")\nprint("="*50)\n'

### Integridad de Datos — Resumen

#### Valores faltantes (NaN)

**Total de valores NaN en el DataFrame:** **4854**

**Top 10 columnas con más valores NaN:**

| Price     | Ticker  | NaN Count |
|-----------|---------|-----------|
| Close     | AENA.MC | 523 |
| Adj Close | AENA.MC | 523 |
| Open      | AENA.MC | 523 |
| Volume    | AENA.MC | 523 |
| Low       | AENA.MC | 523 |
| High      | AENA.MC | 523 |
| Adj Close | ABNB    | 16  |
| Adj Close | AAPL    | 16  |
| Adj Close | COST   | 16  |
| Adj Close | ADBE   | 16  |


#### Filas duplicadas

- **Número de filas duplicadas:** **0**

#### Fechas faltantes (6)

```text
2024-03-29
2024-12-25
2025-01-01
2025-04-18
2025-12-25
2026-01-01

#### Idea rápida

- `AENA.MC` concentra **la mayoría de los NaN** faltan datos históricos.
- No hay filas duplicadas 
- Las fechas faltantes coinciden con **festivos de mercado** (totalmente normal en datos financieros).

Antes de imputar o eliminar NaN por ticker, seguiremos haciendo un estudio básico de los datos ya que puede ser muy precipitado antes de calcular los parámetros históricos

A lo largo del procesamiento vamos a seguir una serie de pasos:

1. Filtrado por fecha de valoración
2. Seleccionar `Adj Close` y calcular $S_{0}$
3. Cálculo de retornos históricos
4. Estimación de volatilidad histórica anualizada
5. Estimación del dividend yield ($d$)

	

In [5]:
from PreprocesamientoIBEX35 import PreprocesamientoIBEX35

prep = PreprocesamientoIBEX35(df_hist)
df_filtrado = prep.filtrar_fecha("2026-01-21")
adj_close, S0 = prep.seleccionar_adj_close()
log_returns = prep.calcular_retornos_log(window=100, min_obs=30)
sigma_daily, sigma_annual = prep.estimar_volatilidad()


sigma_completa = prep.asignar_volatilidad_proxy(sigma_annual, min_obs=30, metodo="promedio")


⚠️ Ticker AENA.MC tiene menos de 30 precios.
⚠️ Ticker AENA.MC tiene 8 precios (<30). Se asignará volatilidad proxy.
✅ AENA.MC asignado volatilidad proxy (promedio): 0.2781


In [6]:
sigma_completa

AAPL       0.205882
ABNB       0.237392
ACX.MC     0.227304
ADBE       0.285543
ANA.MC     0.260562
BBVA.MC    0.241136
COST       0.187713
ELE.MC     0.162122
FER.MC     0.179203
GEHC       0.297028
IBE.MC     0.114885
IDR.MC     0.409032
INTC       0.646974
ITX.MC     0.258381
LOG.MC     0.132280
META       0.299988
MNST       0.215988
NFLX       0.311216
NVDA       0.348827
PEP        0.190023
REP.MC     0.248084
SAN.MC     0.223942
SCYR.MC    0.205988
TEF.MC     0.281578
TSLA       0.462009
WBD        0.598359
AENA.MC    0.278132
dtype: float64

In [7]:


div_yield = prep.calcular_dividend_yield()
print(div_yield)

⚠️ No se pudo obtener dividendos para AAPL: Invalid comparison between dtype=datetime64[s, America/New_York] and Timestamp
⚠️ ABNB no tiene dividendos en yfinance.
⚠️ No se pudo obtener dividendos para ACX.MC: Invalid comparison between dtype=datetime64[s, Europe/Madrid] and Timestamp
⚠️ No se pudo obtener dividendos para ADBE: Invalid comparison between dtype=datetime64[s, America/New_York] and Timestamp
⚠️ AENA.MC no tiene dividendos en yfinance.
⚠️ No se pudo obtener dividendos para ANA.MC: Invalid comparison between dtype=datetime64[s, Europe/Madrid] and Timestamp
⚠️ No se pudo obtener dividendos para BBVA.MC: Invalid comparison between dtype=datetime64[s, Europe/Madrid] and Timestamp
⚠️ No se pudo obtener dividendos para COST: Invalid comparison between dtype=datetime64[s, America/New_York] and Timestamp
⚠️ No se pudo obtener dividendos para ELE.MC: Invalid comparison between dtype=datetime64[s, Europe/Madrid] and Timestamp
⚠️ No se pudo obtener dividendos para FER.MC: Invalid com


# Calcular dividend yield
div_yield = ibex.calcular_dividend_yield()
print(div_yield)

In [8]:
fecha_valoracion = pd.Timestamp("2026-01-21")
df_filtrado = df_hist.loc[df_hist.index <= fecha_valoracion]
print(df_filtrado.index.min(), "→", df_filtrado.index.max())


2024-01-02 00:00:00 → 2026-01-21 00:00:00


In [9]:

adj_close = df_filtrado["Adj Close"]

S0 = adj_close.iloc[-1]

print("Precio inicial S0 por activo:")
print(S0)


Precio inicial S0 por activo:
Ticker
AAPL       247.649994
ABNB       133.589996
ACX.MC      12.890000
ADBE       294.230011
AENA.MC     25.200001
ANA.MC     176.600006
BBVA.MC     20.860001
COST       982.859985
ELE.MC      30.309999
FER.MC      56.680000
GEHC        81.099998
IBE.MC      18.334999
IDR.MC      53.849998
INTC        54.250000
ITX.MC      55.459999
LOG.MC      30.780001
META       612.960022
MNST        81.599998
NFLX        85.360001
NVDA       183.320007
PEP        146.740005
REP.MC      16.139999
SAN.MC      10.324000
SCYR.MC      3.918000
TEF.MC       3.236000
TSLA       431.440002
WBD         28.530001
Name: 2026-01-21 00:00:00, dtype: float64


In [ ]:
# Supongamos que tienes tu df_options y S0_dict listos
from PreprocesamientoOpciones import PreprocesamientoOpciones

df = pd.read_csv("../data/option_chains_all.csv")
proc_opciones = PreprocesamientoOpciones(df, S0)
proc_opciones.convertir_fechas()
proc_opciones.filtrar_vencimiento([21, 52])
proc_opciones.separar_call_put()
proc_opciones.seleccionar_atm()
proc_opciones.calcular_precio_mercado()
proc_opciones.filtrar_liquidez()
proc_opciones.limpiar_nans()
resumen = proc_opciones.resumen()
print(resumen)


{'num_opciones': 6, 'tickers': array(['AAPL', 'NVDA', 'INTC', 'TSLA', 'META', 'NFLX'], dtype=object), 'fechas_expiry': array([Timestamp('2026-08-21 00:00:00'), Timestamp('2027-03-19 00:00:00')],
      dtype=object)}
